# 01 — VinDr-CXR Data Exploration

This notebook explores the dataset before training: split sizes, per-class prevalence, and image/box visualization.

Set `data.root` in `configs/base.yaml` to your VinDr-CXR download first, or pass `--root` here.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))

import matplotlib.pyplot as plt
from src.utils.config import load_config, apply_overrides
from src.data.split import create_splits, load_split_ids, official_split_ids
from src.data.annotations import (
    concepts_from_boxes,
    image_labels_to_multihot,
    load_boxes,
    load_image_labels,
)
from src.data.dataset import VinDrCXRDataset
from src.data.transforms import build_transform, unnormalize
import torch
import numpy as np
import pandas as pd

In [ ]:
cfg = load_config("configs/base.yaml")
apply_overrides(cfg, [])  # or e.g. ["data.root=/path/to/vindr-cxr"]
ROOT = Path(cfg.data.root)
ROOT

In [ ]:
if not (ROOT / "images").exists():
    print("Dataset not found at", ROOT)
    print("Set data.root in configs/base.yaml and run scripts/prepare_data.py")
else:
    splits = create_splits(ROOT, "data/splits", val_fraction=cfg.data.val_fraction, seed=cfg.data.split_seed)
    for name, ids in splits.items():
        print(f"{name:5s}: {len(ids)} images")

In [ ]:
if (ROOT / "images").exists():
    boxes = load_boxes(ROOT / "annotations" / "train.csv")
    labels = load_image_labels(ROOT / "annotations" / "image_labels_train.csv")
    all_ids = list(load_split_ids("data/splits")["train"])

    concept_df = concepts_from_boxes(boxes, list(cfg.concepts), image_ids=all_ids)
    diag_df = image_labels_to_multihot(labels, list(cfg.diagnoses), image_ids=all_ids)

    print("--- Concept prevalence (train split) ---")
    display(concept_df.mean().sort_values(ascending=False).round(4).to_frame("prevalence"))
    print("--- Diagnosis prevalence (train split) ---")
    display(diag_df.mean().sort_values(ascending=False).round(4).to_frame("prevalence"))

In [ ]:
if (ROOT / "images").exists():
    transform = build_transform(cfg.data.image_size, train=False)
    ds = VinDrCXRDataset(
        ROOT, all_ids[:64], list(cfg.concepts), list(cfg.diagnoses),
        split="train", image_size=cfg.data.image_size, transform=transform,
    )
    print("items:", len(ds))
    print("sample keys:", list(ds[0].keys()))

In [ ]:
if (ROOT / "images").exists():
    fig, axes = plt.subplots(2, 3, figsize=(12, 8))
    for ax, i in zip(axes.ravel(), [0, 1, 2, 3, 4, 5]):
        item = ds[i]
        img = unnormalize(item["image"]).squeeze(0).numpy().transpose(1, 2, 0)
        ax.imshow(img, cmap="gray")
        present = [c for c, v in zip(cfg.concepts, item["concepts"].tolist()) if v]
        ax.set_title(f"{item['image_id']}\n{', '.join(present) if present else 'no findings'}", fontsize=8)
        ax.axis("off")
        for box in item["boxes"]:
            import matplotlib.patches as patches
            h, w = img.shape[:2]
            rect = patches.Rectangle(
                (box.x_min * w, box.y_min * h), box.width * w, box.height * h,
                linewidth=1, edgecolor="r", facecolor="none",
            )
            ax.add_patch(rect)
    plt.tight_layout()
    plt.show()

## Next steps

1. `python scripts/prepare_data.py --config configs/base.yaml`
2. `python scripts/train_blackbox.py --config configs/base.yaml --config configs/blackbox.yaml`
3. `python scripts/train_cbm.py --config configs/base.yaml --config configs/cbm.yaml`
4. `python scripts/evaluate.py --checkpoint outputs/checkpoints/*/best_model.pth --split test`